# Feature Engineering — Used Car Dynamic Pricing

This notebook transforms the raw training data into a modeling-ready table: cleaning, date and business features, brand/model aggregations, anonymous-feature screening, and export to `data/processed/train_fe.csv`.

**Input:** `data/raw/used_car_train_20200313.csv`  
**Output:** `data/processed/train_fe.csv`

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# 项目根目录
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "used_car_train_20200313.csv"
OUT_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = OUT_DIR / "train_fe.csv"
IMPORTANCE_REPORT_PATH = REPORT_DIR / "feature_importance_candidates.csv"

RANDOM_STATE = 42
POWER_CAP = 600

## Load raw data

Space-separated file. We load all columns as-is first; `notRepairedDamage` is cleaned explicitly in the next section.

In [2]:
df = pd.read_csv(DATA_PATH, sep=" ")
print(f"Shape: {df.shape}")
df.head(3)

Shape: (150000, 31)


,SaleID,name,regDate,model,brand,bodyType,fuelType,gearbox,power,kilometer,...,v_5,v_6,v_7,v_8,v_9,v_10,v_11,v_12,v_13,v_14
0,0,736,20040402,30.0,6,1.0,0.0,0.0,60,12.5,...,0.235676,0.101988,0.129549,0.022816,0.097462,-2.881803,2.804097,-2.420821,0.795292,0.914762
1,1,2262,20030301,40.0,1,2.0,0.0,0.0,0,15.0,...,0.264777,0.121004,0.135731,0.026597,0.020582,-4.900482,2.096338,-1.030483,-1.722674,0.245522
2,2,14874,20040403,115.0,15,1.0,0.0,0.0,163,12.5,...,0.251410,0.114912,0.165147,0.062173,0.027075,-4.846749,1.803559,1.565330,-0.832687,-0.229963


## Data cleaning

### 1. `notRepairedDamage`: replace `"-"` with NaN

The field encodes whether the vehicle has unrepaired damage (`0.0` / `1.0`). A dash means **unknown** and must not be treated as a category.

In [3]:
df["notRepairedDamage"] = df["notRepairedDamage"].replace("-", np.nan)
print(df["notRepairedDamage"].value_counts(dropna=False).head())

notRepairedDamage
0.0    111361
NaN     24324
1.0     14315
Name: count, dtype: int64


### 2. Fill missing values

| Column | Strategy | Rationale |
|--------|----------|----------|
| `bodyType`, `fuelType`, `gearbox`, `notRepairedDamage` | **Mode** | Low-cardinality categoricals; mode is stable on 150k rows |
| `power` | **Median** | Robust to extreme engine-power errors before capping |

In [4]:
mode_cols = ["bodyType", "fuelType", "gearbox", "notRepairedDamage"]

for col in mode_cols:
    fill_val = df[col].mode(dropna=True).iloc[0]
    n_missing = df[col].isna().sum()
    df[col] = df[col].fillna(fill_val)
    print(f"{col}: filled {n_missing:,} with mode={fill_val}")

power_median = df["power"].median()
n_power_missing = df["power"].isna().sum()
df["power"] = df["power"].fillna(power_median)
print(f"power: filled {n_power_missing:,} with median={power_median}")

assert df[mode_cols + ["power"]].isna().sum().sum() == 0

bodyType: filled 4,506 with mode=0.0
fuelType: filled 8,680 with mode=0.0
gearbox: filled 5,981 with mode=0.0
notRepairedDamage: filled 24,324 with mode=0.0
power: filled 0 with median=110.0


### 3. Handle outliers: cap `power` at 600

EDA shows impossible values (e.g. thousands). Capping preserves rank for most vehicles while limiting leverage from bad records.

In [5]:
n_capped = (df["power"] > POWER_CAP).sum()
df["power"] = df["power"].clip(upper=POWER_CAP)
print(f"Capped {n_capped:,} rows to power <= {POWER_CAP}")

Capped 143 rows to power <= 600


## Date feature engineering

### 4. Convert `regDate` and `creatDate` to datetime

Stored as integers `YYYYMMDD` (registration date vs listing creation date on the platform).

In [6]:
def yyyymmdd_to_datetime(series: pd.Series) -> pd.Series:
    """将 YYYYMMDD 整数列转为 datetime"""
    return pd.to_datetime(series.astype(str), format="%Y%m%d", errors="coerce")


df["regDate_dt"] = yyyymmdd_to_datetime(df["regDate"])
df["creatDate_dt"] = yyyymmdd_to_datetime(df["creatDate"])

invalid_reg = df["regDate_dt"].isna().sum()
invalid_creat = df["creatDate_dt"].isna().sum()
print(f"Invalid regDate: {invalid_reg:,}, invalid creatDate: {invalid_creat:,}")

Invalid regDate: 11,347, invalid creatDate: 0


### 5. Derived date features

| Feature | Definition |
|---------|------------|
| `car_age_days` | Days from registration to listing (`creatDate_dt - regDate_dt`) |
| `car_age_years` | `car_age_days / 365.25` — used for usage intensity |
| `reg_year`, `reg_month` | Calendar parts of registration (seasonality, vintage) |
| `creat_year`, `creat_month` | Calendar parts of listing (market timing) |

In [7]:
df["car_age_days"] = (df["creatDate_dt"] - df["regDate_dt"]).dt.days
df["car_age_years"] = df["car_age_days"] / 365.25

df["reg_year"] = df["regDate_dt"].dt.year
df["reg_month"] = df["regDate_dt"].dt.month
df["creat_year"] = df["creatDate_dt"].dt.year
df["creat_month"] = df["creatDate_dt"].dt.month

# 异常车龄（登记晚于上架）置为缺失，后续分箱会处理
bad_age = df["car_age_days"] < 0
print(f"Negative car_age_days: {bad_age.sum():,}")
df.loc[bad_age, ["car_age_days", "car_age_years"]] = np.nan

df[["car_age_days", "car_age_years", "reg_year", "reg_month", "creat_year", "creat_month"]].describe()

Negative car_age_days: 0


,car_age_days,car_age_years,reg_year,reg_month,creat_year,creat_month
count,138653.000000,138653.000000,138653.000000,138653.000000,150000.000000,150000.000000
mean,4432.082407,12.134380,2003.619518,6.407463,2015.999880,3.161580
std,1953.201975,5.347576,5.342735,3.347310,0.010954,0.380709
min,88.000000,0.240931,1991.000000,1.000000,2015.000000,1.000000
25%,2941.000000,8.052019,2000.000000,4.000000,2016.000000,3.000000
50%,4418.000000,12.095825,2004.000000,6.000000,2016.000000,3.000000
75%,5927.000000,16.227242,2008.000000,9.000000,2016.000000,3.000000
max,9222.000000,25.248460,2015.000000,12.000000,2016.000000,12.000000


## Business features

### 6. Usage, power, and age bins

| Feature | Definition |
|---------|------------|
| `km_per_year` | `kilometer / car_age_years` — annualized mileage (proxy for wear); denominator floored at 0.25 years |
| `power_bin` | Binned engine power after cap — nonlinear effect for tree/linear models |
| `car_age_bin` | Binned vehicle age — captures depreciation buckets |

In [8]:
age_years_safe = df["car_age_years"].clip(lower=0.25)
df["km_per_year"] = df["kilometer"] / age_years_safe

power_edges = [0, 60, 100, 140, 200, 300, POWER_CAP]
df["power_bin"] = pd.cut(
    df["power"],
    bins=power_edges,
    labels=["p0_60", "p60_100", "p100_140", "p140_200", "p200_300", "p300_600"],
    include_lowest=True,
)

age_edges = [0, 1, 3, 5, 8, 12, 20, 50]
df["car_age_bin"] = pd.cut(
    df["car_age_years"],
    bins=age_edges,
    labels=["a0_1", "a1_3", "a3_5", "a5_8", "a8_12", "a12_20", "a20_50"],
    include_lowest=True,
)

df[["km_per_year", "power_bin", "car_age_bin"]].head()

,km_per_year,power_bin,car_age_bin
0,1.041192,p0_60,a12_20
1,1.151724,p0_60,a12_20
2,1.041904,p140_200,a8_12
3,0.768947,p140_200,a12_20
4,1.192848,p60_100,a3_5


## Brand aggregation features

### 7. Brand-level price statistics

Computed on the **training set** grouped by `brand`:

- `brand_price_mean` — average transaction price for the brand  
- `brand_price_median` — robust central price  
- `brand_price_std` — price dispersion (luxury vs mass-market spread)  
- `brand_price_max` — upper tail of brand listings  

**Note:** These use the target `price` and introduce **target leakage** if applied to the same rows used to fit the model. For production or validation, use out-of-fold encoding or holdout brand stats. Here we build the training feature matrix as specified.

In [9]:
brand_price_agg = (
    df.groupby("brand", observed=True)["price"]
    .agg(
        brand_price_mean="mean",
        brand_price_median="median",
        brand_price_std="std",
        brand_price_max="max",
    )
    .reset_index()
)

df = df.merge(brand_price_agg, on="brand", how="left")
df[["brand", "price"] + [c for c in brand_price_agg.columns if c != "brand"]].head()

,brand,price,brand_price_mean,brand_price_median,brand_price_std,brand_price_max
0,6,1850,3611.840266,1800.0,4681.293524,59900
1,1,3600,9273.311947,6499.0,9369.631497,99900
2,15,6222,9858.582990,8500.0,5425.058140,45000
3,10,2400,8470.804197,5400.0,8988.307535,98000
4,5,5200,3306.349411,2300.0,3343.624586,31500


## Model aggregation features

### 8. Model-level power statistics

| Feature | Definition |
|---------|------------|
| `model_power_mean` | Mean `power` for the model code (typical engine class) |
| `model_power_std` | Spread of power within model — specification variance |
| `model_count` | Number of listings per model — popularity / sample reliability |

In [10]:
model_power_agg = (
    df.groupby("model", observed=True)["power"]
    .agg(
        model_power_mean="mean",
        model_power_std="std",
        model_count="count",
    )
    .reset_index()
)

df = df.merge(model_power_agg, on="model", how="left")
df[["model", "power", "model_power_mean", "model_power_std", "model_count"]].head()

,model,power,model_power_mean,model_power_std,model_count
0,30.0,60,64.700256,31.990438,2342.0
1,40.0,0,140.859396,52.495875,4502.0
2,115.0,163,137.343042,40.963832,927.0
3,109.0,193,259.880829,102.059068,386.0
4,110.0,68,57.186004,23.085682,543.0


## Anonymous feature exploration (`v_0`–`v_14`)

### 9. Correlation with `price`

Pearson correlation on the training sample. Strong linear association suggests high value for linear models and GBDT; check multicollinearity among `v_*` before linear regression.

In [11]:
v_cols = [f"v_{i}" for i in range(15)]

v_price_corr = (
    df[v_cols + ["price"]]
    .corr(numeric_only=True)["price"]
    .drop("price")
)
v_price_corr_abs = v_price_corr.abs().sort_values(ascending=False)

v_corr_report = pd.DataFrame(
    {
        "pearson_r": v_price_corr.reindex(v_price_corr_abs.index),
        "abs_pearson_r": v_price_corr_abs,
    }
)
v_corr_report

,pearson_r,abs_pearson_r
v_3,-0.730946,0.730946
v_12,0.692823,0.692823
v_8,0.685798,0.685798
v_0,0.628397,0.628397
v_11,-0.275320,0.275320
v_10,-0.246175,0.246175
v_9,-0.206205,0.206205
v_5,0.164317,0.164317
v_4,-0.147085,0.147085
v_2,0.085322,0.085322


### 10. Feature importance candidate report

We rank **all numeric candidates** (raw, engineered, `v_*`, brand price stats) by absolute Pearson correlation with `price`. This is a cheap screening step—not a substitute for model-based importance (SHAP, permutation) but useful to prioritize features for baseline models.

Report saved to `reports/feature_importance_candidates.csv`.

In [12]:
engineered_num = [
    "car_age_days",
    "car_age_years",
    "reg_year",
    "reg_month",
    "creat_year",
    "creat_month",
    "km_per_year",
    "brand_price_mean",
    "brand_price_median",
    "brand_price_std",
    "brand_price_max",
    "model_power_mean",
    "model_power_std",
    "model_count",
]

raw_num = [
    "power",
    "kilometer",
    "model",
    "brand",
    "bodyType",
    "fuelType",
    "gearbox",
    "regionCode",
    "seller",
    "offerType",
    "name",
]

candidate_cols = list(dict.fromkeys(raw_num + engineered_num + v_cols))
candidate_cols = [c for c in candidate_cols if c in df.columns]

full_corr = df[candidate_cols + ["price"]].corr(numeric_only=True)["price"].drop("price")

importance_report = (
    pd.DataFrame(
        {
            "feature": full_corr.index,
            "pearson_r": full_corr.values,
            "abs_pearson_r": np.abs(full_corr.values),
        }
    )
    .sort_values("abs_pearson_r", ascending=False)
    .reset_index(drop=True)
)
importance_report["feature_group"] = np.where(
    importance_report["feature"].isin(v_cols),
    "anonymous_v",
    np.where(
        importance_report["feature"].str.startswith("brand_price"),
        "brand_target_agg",
        np.where(
            importance_report["feature"].str.startswith("model_"),
            "model_agg",
            np.where(
                importance_report["feature"].isin(engineered_num),
                "engineered",
                "raw",
            ),
        ),
    ),
)

importance_report.to_csv(IMPORTANCE_REPORT_PATH, index=False)
print(f"Saved: {IMPORTANCE_REPORT_PATH}")
importance_report.head(20)

Saved: reports/feature_importance_candidates.csv


,feature,pearson_r,abs_pearson_r,feature_group
0,v_3,-0.730946,0.730946,anonymous_v
1,v_12,0.692823,0.692823,anonymous_v
2,v_8,0.685798,0.685798,anonymous_v
3,v_0,0.628397,0.628397,anonymous_v
4,reg_year,0.610874,0.610874,engineered
5,car_age_days,-0.610510,0.610510,engineered
6,car_age_years,-0.610510,0.610510,engineered
7,power,0.556400,0.556400,raw
8,kilometer,-0.440519,0.440519,raw
9,brand_price_mean,0.435850,0.435850,brand_target_agg


## Final dataset

### 11. Save processed training data

Export includes original columns plus all engineered fields. Categorical bins are stored as strings for CSV compatibility.

In [13]:
new_features = [
    "regDate_dt",
    "creatDate_dt",
    "car_age_days",
    "car_age_years",
    "reg_year",
    "reg_month",
    "creat_year",
    "creat_month",
    "km_per_year",
    "power_bin",
    "car_age_bin",
    "brand_price_mean",
    "brand_price_median",
    "brand_price_std",
    "brand_price_max",
    "model_power_mean",
    "model_power_std",
    "model_count",
]

print(f"Output shape: {df.shape}")
print(f"New columns ({len(new_features)}): {new_features}")
remaining_nulls = df.isna().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print(f"Remaining nulls:\n{remaining_nulls}")

Output shape: (150000, 49)
New columns (18): ['regDate_dt', 'creatDate_dt', 'car_age_days', 'car_age_years', 'reg_year', 'reg_month', 'creat_year', 'creat_month', 'km_per_year', 'power_bin', 'car_age_bin', 'brand_price_mean', 'brand_price_median', 'brand_price_std', 'brand_price_max', 'model_power_mean', 'model_power_std', 'model_count']
Remaining nulls:
model                   1
regDate_dt          11347
car_age_days        11347
car_age_years       11347
reg_year            11347
reg_month           11347
km_per_year         11347
car_age_bin         11347
model_power_mean        1
model_power_std         2
model_count             1
dtype: int64


In [14]:
df.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")
print(f"File size (MB): {OUT_PATH.stat().st_size / 1e6:.2f}")

Saved: data/processed/train_fe.csv
File size (MB): 81.33


## Summary

| Step | Action |
|------|--------|
| Cleaning | `"-"` → NaN in `notRepairedDamage`; mode/median impute; cap `power` at 600 |
| Dates | `regDate_dt`, `creatDate_dt`, age and calendar parts |
| Business | `km_per_year`, `power_bin`, `car_age_bin` |
| Aggregations | Brand price stats; model power stats + count |
| Screening | `v_*` correlation + ranked candidate report |
| Export | `data/processed/train_fe.csv` |

**Next steps:** encode categoricals, OOF target encoding for brand stats, train/validation split, and baseline models (e.g. LightGBM) using top features from the importance report.